# EmotWen — Full Pipeline

Fine-tunes **Qwen 3.5 (0.8B)** into an empathetic journal companion chatbot. Run all cells top-to-bottom.

**Stages:** Data Prep → SFT (Stage 1+2) → Evaluation → GRPO (conditional)

**Kaggle setup:** Enable GPU T4 x2 + Internet. Add `WANDB_API_KEY` to Secrets (Add-ons → Secrets).

In [ ]:
%%capture
# ── Install Unsloth + TRL stack ───────────────────────────────────────────────
import os, importlib.util, sys
!pip install --upgrade -qqq uv
if importlib.util.find_spec('torch') is None or 'COLAB_' in ''.join(os.environ.keys()) or os.path.exists('/kaggle/working') or os.path.exists('/workspace'):
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = 'numpy'; _pil = 'pillow'
    # Uninstall torchvision to avoid version mismatch with torch==2.8.0
    !pip uninstall -qqq -y torchvision torchaudio 2>/dev/null || true
    !uv pip install -qqq \
        'torch==2.8.0' 'triton>=3.3.0' {_numpy} {_pil} bitsandbytes 'xformers==0.0.32.post2' \
        'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo' \
        'unsloth[base] @ git+https://github.com/unslothai/unsloth'
elif importlib.util.find_spec('unsloth') is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers 'trl==0.22.2' unsloth unsloth_zoo
!uv pip install 'transformers==5.2.0'
!uv pip install --no-build-isolation flash-linear-attention 'causal_conv1d==1.6.0'
!uv pip install wandb datasets nltk

In [ ]:
# ── Restart kernel to avoid PyArrow / binary incompatibility errors ───────────
# The kernel will restart automatically. After it restarts, run all remaining
# cells from "Set up repo" onward — do NOT re-run the install cell above.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# ── Set up repo ───────────────────────────────────────────────────────────────
import os

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'COLAB_' in ''.join(os.environ.keys())
IS_VAST   = os.path.exists('/workspace') and not IS_KAGGLE and not IS_COLAB

if IS_KAGGLE:
    REPO_DIR = '/kaggle/working/emotwen-3.5-finetune'
elif IS_COLAB:
    REPO_DIR = '/content/emotwen-3.5-finetune'
elif IS_VAST:
    REPO_DIR = '/workspace/emotwen-3.5-finetune'
else:
    REPO_DIR = os.path.dirname(os.path.abspath(''))

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/afonsomota/emotwen-3.5-finetune.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Repo dir: {REPO_DIR}')
import nltk
nltk.download('punkt_tab', quiet=True)

In [ ]:
# ── W&B login ─────────────────────────────────────────────────────────────────
import os, wandb

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')

if os.environ.get('WANDB_API_KEY'):
    wandb.login(key=os.environ['WANDB_API_KEY'])
else:
    wandb.login()  # anonymous / prompts if interactive

## Stage 1 — Data Preparation

Loads, filters, and formats all training data. Saves to `data/`.

In [ ]:
# ── Config overrides (edit this cell to customise) ────────────────────────────
CONFIG = {
    # W&B
    'wandb_project': 'emotwen-journal-chat',
    'run_name': 'data_prep_v1',

    # Dataset sizes (set to None to use all available)
    'max_empathetic': 15000,
    'max_daily_dialog': 6000,
    'max_go_emotions_synthetic': 5000,
    'max_counsel_chat': 2000,

    # RAG simulation: fraction of synthetic examples with journal context injected
    'rag_injection_fraction': 0.30,

    # Held-out eval set size
    'eval_holdout_size': 200,

    # Override save dirs to Drive for persistence across sessions
    # 'train_save_dir': '/content/drive/MyDrive/emotwen/data/sft_train',
    # 'val_save_dir':   '/content/drive/MyDrive/emotwen/data/sft_val',
    # 'eval_save_dir':  '/content/drive/MyDrive/emotwen/data/eval_200',
}

In [ ]:
# ── Run pipeline ──────────────────────────────────────────────────────────────
from src.data_prep import run

results = run(CONFIG)
print('\n── Results ──────────────────────────────────────────────')
for k, v in results.items():
    if not isinstance(v, dict):
        print(f'  {k}: {v}')
print('\nSources breakdown:')
for k, v in results['sources_breakdown'].items():
    print(f'  {k}: {v}')

In [ ]:
# ── Inspect sample conversations ──────────────────────────────────────────────
from datasets import load_from_disk
import os

train_ds = load_from_disk(os.path.join(REPO_DIR, 'data', 'sft_train'))

print(f'Train set size: {len(train_ds)}')
print('\n── Sample conversations ─────────────────────────────────')
for i in [0, 100, 500]:
    ex = train_ds[i]
    print(f'\n[Example {i}] source={ex["source"]}')
    for msg in ex['messages']:
        role = msg['role'].upper()
        content = msg['content'][:200].replace('\n', ' ')
        print(f'  {role}: {content}')
    print()

## Stage 2 — SFT Training

Two-stage supervised fine-tuning:
- **Stage 1** (tone): empathetic_dialogues + daily_dialog
- **Stage 2** (domain): synthetic journal conversations at lower LR

In [ ]:
# ── Config overrides (edit this cell) ────────────────────────────────────────
CONFIG = {
    # W&B
    'wandb_project': 'emotwen-journal-chat',
    'run_name_s1': 'sft_stage1_v1',
    'run_name_s2': 'sft_stage2_v1',

    # Which stage(s) to run: '1', '2', or 'both'
    'stage': 'both',

    # Stage 1 steps (set lower for quick debug)
    #'max_steps': 2,  # uncomment for quick test

    # Override output dirs to Drive for persistence
    # 'output_dir': '/content/drive/MyDrive/emotwen/outputs/sft_stage1',  # stage 1
}

In [ ]:
# ── Run SFT ─────────────────────────────────────────────
from src.train_sft import run

results = run(CONFIG)
print('\n── SFT Results ──────────────────────────────────────────')
for k, v in results.items():
    print(f'  {k}: {v}')

In [ ]:
# ── GPU memory summary ────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    
    used = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Peak GPU memory: {used:.2f} GB / {total:.2f} GB ({100*used/total:.1f}%)')

## Stage 3 — Evaluation

Evaluates on held-out `eval_200`. **Decision gate:** if >15% responses exceed 5 sentences → run Stage 4.

In [ ]:
# ── Optional: set API keys for LLM-as-judge ──────────────────────────────────
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

In [ ]:
# ── Config overrides ──────────────────────────────────────────────────────────
CONFIG = {
    'wandb_project': 'emotwen-journal-chat',
    'run_name': 'eval_post_sft_v1',

    # Path to SFT adapter (stage 2)
    'adapter_path': os.path.join(REPO_DIR, 'outputs', 'sft_stage2'),

    # Eval samples (up to 200)
    'n_samples': 200,


    # GRPO trigger threshold
    'grpo_trigger_pct': 0.15,
}

In [ ]:
# ── Run evaluation ─────────────────────────────────────────────────────────────
from src.evaluate import run

results = run(CONFIG)

print('\n── Summary ──────────────────────────────────────────────')
print(f'  Responses in range (2–5 sentences): {results["pct_in_range"]:.1%}')
print(f'  Responses >5 sentences:             {results["pct_over_5"]:.1%}')
print(f'  Advice rate:                        {results["advice_rate"]:.1%}')
print(f'  Emotion alignment:                  {results["emotion_alignment"]:.1%}')
print(f'  Perplexity (mean NLL/token):        {results["perplexity"]:.4f}')
print(f'  LLM empathy score (reflection):     {results["llm_reflection_avg"]}')
print(f'  LLM no-advice score:                {results["llm_no_advice_avg"]}')
print(f'  "Let me explain:" accuracy:         {results["let_me_explain_ok_rate"]}')
print()
print(f'  ➜ Run GRPO: {"YES ⚠" if results["grpo_needed"] else "NO ✓"}')

## Stage 4 — GRPO Length Control *(conditional)*

> Only run if Stage 3 reported `grpo_needed=True`.

Reward functions: `length_reward` (2–5 sentences) + `advice_penalty_reward`.

In [ ]:
# ── Config overrides ──────────────────────────────────────────────────────────
CONFIG = {
    'wandb_project': 'emotwen-journal-chat',
    'run_name': 'grpo_v1',

    # SFT adapter to start from
    'sft_adapter_path': os.path.join(REPO_DIR, 'outputs', 'sft_stage2'),

    # GRPO output dirs
    'output_dir': os.path.join(REPO_DIR, 'outputs', 'grpo_adapter'),
    'final_merged_dir': os.path.join(REPO_DIR, 'outputs', 'final_merged'),

    # Set skip_if_not_needed=True to auto-abort if eval said grpo_needed=False
    'skip_if_not_needed': False,

    # Training steps (reduce for quick debug)
    # 'max_steps': 20,

    # Number of prompts to train on
    'n_grpo_prompts': 400,
}

In [ ]:
# ── Run GRPO ──────────────────────────────────────────────────────────────────
from src.train_grpo import run

results = run(CONFIG)

if results.get('grpo_skipped'):
    print('GRPO was skipped (model already within length bounds).')
else:
    print('\n── GRPO Results ─────────────────────────────────────────')
    print(f'  Pre-GRPO  pct_in_range: {results["pre_grpo_pct_in_range"]}')
    print(f'  Post-GRPO pct_in_range: {results["post_grpo_pct_in_range"]:.1%}')
    print(f'  Post-GRPO advice_rate:  {results["post_grpo_advice_rate"]:.1%}')
    print(f'  Still needs GRPO:       {results["post_grpo_grpo_still_needed"]}')
    print(f'  Final merged model:     {results["final_merged_dir"]}')

In [ ]:
# ── GPU memory summary ────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    used = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Peak GPU memory: {used:.2f} GB / {total:.2f} GB ({100*used/total:.1f}%)')